## Training Embeddings with Gensim

This notebook is a part of [AI for Beginners Curriculum](http://aka.ms/ai-beginners)

In the [CBoW notebooks](CBoW-PyTorch.ipynb) we wrote the Word2Vec training loop ourselves, which is a great way to understand what is going on. In practice, however, we usually reach for a library. [Gensim](https://radimrehurek.com/gensim/) can train Word2Vec and FastText embeddings on millions of words in a few lines of code.

In this notebook we will:

1. Train **Word2Vec** with both architectures - CBoW and skip-gram - and see what the main hyper-parameters do.
2. Train **FastText** and see how it deals with words it has never seen.
3. Explore the embeddings: similar words, analogies and a 2D picture of the embedding space.
4. **Save** the vectors, **load** them back, and use them to initialize the embedding layer of a news classifier.

We will use the text of the **AG News** dataset (the same one we used in the previous lessons) as our corpus. Training runs on CPU, and the whole notebook should take about 5 minutes.

In [ ]:
import os
import re
import time
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from gensim.models import Word2Vec, FastText, KeyedVectors
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
workers = min(4, os.cpu_count() or 1)   # number of CPU threads used by gensim

### Loading the Text

AG News is distributed as simple CSV files with three columns: class number, title and description. We download them once into the `data` folder.

> **Note:** Here we read the CSV files directly, so we do not need `torchtext`. It is the same data that `torchtext.datasets.AG_NEWS` gives you.

In [ ]:
os.makedirs('data', exist_ok=True)

base_url = 'https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/'
for name in ['train.csv', 'test.csv']:
    path = os.path.join('data', 'ag_news_' + name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(base_url + name, path)

columns = ['label', 'title', 'description']
train_df = pd.read_csv('data/ag_news_train.csv', header=None, names=columns)
test_df = pd.read_csv('data/ag_news_test.csv', header=None, names=columns)
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

print(len(train_df), 'training and', len(test_df), 'test articles')
train_df.head()

Word2Vec is trained on **unlabeled** text, so we only need the words. Gensim expects the corpus as a list of *sentences*, where each sentence is a list of tokens. We will treat each news article (title + description) as one "sentence" and use a very simple tokenizer: lower-case the text and keep the sequences of letters and digits.

We train only on the training articles, and **not** on the test set. We will need the test set later to evaluate a classifier.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

sentences = [tokenize(title + ' ' + descr)
             for title, descr in zip(train_df.title, train_df.description)]

print(sentences[0])
print('Total number of tokens:', sum(len(s) for s in sentences))

## Word2Vec: CBoW and Skip-Gram

Training a model is a single call. Let's look at the most important parameters:

| Parameter | What it does | Typical values |
|-----------|--------------|----------------|
| `vector_size` | Number of dimensions of the embedding vectors | 100 - 300 |
| `window` | Maximum distance between the word we look at and its context words | 2 - 10 |
| `min_count` | Words that appear fewer times than this are ignored (they are too rare to learn anything useful) | 5 - 20 |
| `sg` | Architecture: `0` = CBoW (predict the word from its context), `1` = skip-gram (predict the context from the word) | 0 or 1 |
| `negative` | Number of *negative samples* (see below) | 5 - 20 |
| `epochs` | How many times we go through the corpus | 5 - 20 |

**Negative sampling** is the trick that makes training fast. In our CBoW notebook the network had `vocab_size` outputs, so every training example required computing a score for all the words in the vocabulary. With negative sampling, the model only learns to tell the true context word apart from `negative` randomly chosen "noise" words, which is much cheaper. (Setting `negative=0` and `hs=1` switches gensim to a different technique called hierarchical softmax.)

First, let's train the CBoW version:

In [ ]:
start = time.time()
w2v_cbow = Word2Vec(sentences,
                    vector_size=100,   # embedding dimensions
                    window=5,          # context: 5 words to the left and 5 to the right
                    min_count=5,       # ignore very rare words
                    sg=0,              # 0 = CBoW
                    negative=5,        # negative samples per training example
                    epochs=5,
                    workers=workers,
                    seed=SEED)
print(f'CBoW trained in {time.time() - start:.0f} seconds, vocabulary size: {len(w2v_cbow.wv)}')

Now the skip-gram version - the only change is `sg=1`. Skip-gram creates more training examples (one for each context word), so it is slower, but it is usually better for rare words.

In [ ]:
start = time.time()
w2v_sg = Word2Vec(sentences,
                  vector_size=100,
                  window=5,
                  min_count=5,
                  sg=1,                # 1 = skip-gram
                  negative=5,
                  epochs=5,
                  workers=workers,
                  seed=SEED)
print(f'Skip-gram trained in {time.time() - start:.0f} seconds, vocabulary size: {len(w2v_sg.wv)}')

> **Note:** The trained vectors live in the `model.wv` object (it is a `KeyedVectors` instance). Because gensim uses several threads, the results will be slightly different each time you run the training. For exactly reproducible results you can use `workers=1`, but it is slower.

Each word is now a vector of 100 numbers. Here are the first 10 components for the word *oil*:

In [ ]:
print(w2v_sg.wv['oil'][:10])
print('Shape:', w2v_sg.wv['oil'].shape)

### Similar Words

The `most_similar` function returns the words whose vectors are closest (by cosine similarity) to the given word. Let's compare CBoW and skip-gram:

In [ ]:
for word in ['oil', 'football', 'google', 'iraq']:
    print(f'--- {word} ---')
    for name, model in [('CBoW     ', w2v_cbow), ('skip-gram', w2v_sg)]:
        neighbours = [w for w, _ in model.wv.most_similar(word, topn=6)]
        print(name, neighbours)

We can also ask for the similarity of two words, or find the odd one out:

In [ ]:
print('oil / crude    :', w2v_sg.wv.similarity('oil', 'crude'))
print('oil / football :', w2v_sg.wv.similarity('oil', 'football'))
print('Odd one out    :', w2v_sg.wv.doesnt_match(['soccer', 'tennis', 'baseball', 'stocks']))

### Analogies

Because words are vectors, we can do arithmetic with them. The famous example is *king - man + woman = queen*. Our corpus is news, not a story book, so let's try a geography example instead: *Paris - France + Iraq* should be the capital of Iraq.

In gensim, the words we add are given in `positive`, and the words we subtract in `negative`:

In [ ]:
for name, model in [('CBoW', w2v_cbow), ('skip-gram', w2v_sg)]:
    result = model.wv.most_similar(positive=['paris', 'iraq'], negative=['france'], topn=5)
    print(name, [(w, round(s, 2)) for w, s in result])

✅ Did both models find *Baghdad*? Try some more analogies of your own. Keep in mind that we trained on only 120,000 short news articles (less than 5 million words), while the famous pre-trained embeddings were trained on billions of words - so do not expect every analogy to work.

### The Effect of Hyper-Parameters

Let's see what `window` and `min_count` do. We start with `min_count`. It is cheap to check, because we only need to build the vocabulary, without training:

In [ ]:
for min_count in [1, 5, 20, 100]:
    vocab_only = Word2Vec(min_count=min_count)
    vocab_only.build_vocab(sentences)
    print(f'min_count={min_count:<4} vocabulary size: {len(vocab_only.wv)}')

Most of the words in the corpus are very rare - they appear only a few times, and we cannot learn good vectors for them anyway. Setting `min_count` higher makes the vocabulary (and the model) much smaller.

To see the effect of `window`, we train two small models on a part of the corpus. A small window looks at the immediate neighbours of a word, a large window looks at the whole topic of the sentence:

In [ ]:
for window in [2, 10]:
    model = Word2Vec(sentences[:40000], vector_size=100, window=window, min_count=5,
                     sg=1, epochs=5, workers=workers, seed=SEED)
    print(f'--- window={window} ---')
    for word in ['football', 'oil']:
        print(word, '->', [w for w, _ in model.wv.most_similar(word, topn=8)])

✅ Can you see a difference between the neighbours? Smaller windows tend to give words that can be *substituted* for each other (such as other sports), while larger windows give more words that are *related* to the topic (such as the coach or the prices). The best value depends on what you want to use the embeddings for.

## FastText: Embeddings for Unknown Words

Word2Vec has one weakness: a word that was not in the training corpus does not have a vector at all, and a misspelled word is an unknown word.

[**FastText**](https://fasttext.cc/) fixes this by learning vectors for *character n-grams* (for example, `pre`, `res`, `esi`, ...) and representing a word as the sum of its n-gram vectors. As a result, it can build a vector for any word, even if it has never seen it. The parameters `min_n` and `max_n` control the length of the character n-grams. It is slower to train than Word2Vec, because there is more to learn:

In [ ]:
start = time.time()
ft = FastText(sentences,
              vector_size=100,
              window=5,
              min_count=5,
              sg=0,
              min_n=3, max_n=6,    # character n-grams of length 3 to 6
              epochs=3,
              workers=workers,
              seed=SEED)
print(f'FastText trained in {time.time() - start:.0f} seconds')

Let's try some misspelled words that do not appear in our corpus. Word2Vec raises an error, but FastText still builds a vector for them, and the closest words are the correct spellings:

In [ ]:
word = 'presidnt'
print('In the Word2Vec vocabulary:', word in w2v_sg.wv.key_to_index)
try:
    w2v_sg.wv[word]
except KeyError as e:
    print('Word2Vec:', e)

print()
for word in ['presidnt', 'footbal', 'olympcs']:
    print('FastText:', word, '->', [w for w, _ in ft.wv.most_similar(word, topn=4)])

## Visualizing the Embedding Space

It is hard to imagine a 100-dimensional space, so to see what the embeddings learned, we project them to 2D. We take 12 words for each of the four AG News topics and use two popular techniques:

* **PCA** is a linear projection that preserves as much of the variance as possible.
* **t-SNE** is a non-linear method that tries to keep close points close, so it usually shows clusters better. The picture can change a lot with `perplexity` and the random seed.

In [ ]:
topics = {
    'World':    ['iraq', 'baghdad', 'president', 'minister', 'election', 'israel',
                 'palestinian', 'china', 'russia', 'troops', 'war', 'government'],
    'Sports':   ['football', 'baseball', 'basketball', 'olympic', 'champion', 'coach',
                 'season', 'game', 'league', 'team', 'tennis', 'golf'],
    'Business': ['oil', 'stocks', 'profit', 'shares', 'economy', 'dollar',
                 'market', 'bank', 'company', 'inflation', 'earnings', 'prices'],
    'Sci/Tech': ['software', 'microsoft', 'google', 'internet', 'computer', 'windows',
                 'linux', 'nasa', 'scientists', 'wireless', 'chip', 'security'],
}
colors = ['tab:red', 'tab:green', 'tab:blue', 'tab:orange']

words = [w for ws in topics.values() for w in ws]
vectors = np.array([w2v_sg.wv[w] for w in words])

projections = {
    'PCA': PCA(n_components=2, random_state=SEED).fit_transform(vectors),
    't-SNE': TSNE(n_components=2, perplexity=10, init='pca', random_state=SEED).fit_transform(vectors),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (title, points) in zip(axes, projections.items()):
    start = 0
    for (topic, topic_words), color in zip(topics.items(), colors):
        pts = points[start:start + len(topic_words)]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=topic)
        for (x, y), w in zip(pts, topic_words):
            ax.annotate(w, (x, y), xytext=(3, 3), textcoords='offset points', fontsize=9)
        start += len(topic_words)
    ax.set_title(title)
    ax.legend()
plt.show()

Nobody told the model about the topics, but the words from the same topic mostly end up close to each other, especially in the t-SNE picture.

## Saving and Loading Embeddings

Training embeddings takes time, so we want to save them. Gensim offers several options:

* `model.save(...)` stores the **full model**, so that you can continue training later. It is the biggest file, because it also contains the weights of the output layer.
* `model.wv.save(...)` stores only the **word vectors** (`KeyedVectors`). This is usually all you need to *use* the embeddings.
* `model.wv.save_word2vec_format(...)` writes the vectors in the original Word2Vec format, which other tools can read. Use `binary=False` if you want a human-readable text file.

In [ ]:
w2v_sg.save('data/ag_news_sg.model')
w2v_sg.wv.save('data/ag_news_sg.kv')
w2v_sg.wv.save_word2vec_format('data/ag_news_sg.bin', binary=True)

for f in ['ag_news_sg.model', 'ag_news_sg.kv', 'ag_news_sg.bin']:
    print(f'{f:<18}{os.path.getsize(os.path.join("data", f)) / 1e6:6.1f} MB')

Let's load the vectors back and check that we get exactly the same result:

In [ ]:
wv = KeyedVectors.load('data/ag_news_sg.kv')                                   # gensim format
wv_from_bin = KeyedVectors.load_word2vec_format('data/ag_news_sg.bin', binary=True)  # word2vec format

print(np.allclose(wv['oil'], w2v_sg.wv['oil']), np.allclose(wv_from_bin['oil'], w2v_sg.wv['oil']))
print([w for w, _ in wv.most_similar('oil', topn=5)])

> **Tip:** You do not always have to train your own embeddings. The `gensim.downloader` module can download many pre-trained ones, for example `gensim.downloader.load('glove-wiki-gigaword-100')` returns a `KeyedVectors` object, just like the one we have here. Training your own embeddings is useful when your text is **domain specific** (medical, legal, source code, ...) and general-purpose embeddings do not know your words.

## Using Embeddings in a Neural Network

Now let's put our embeddings to work. We will train a news classifier like the one from the [Embeddings](../14-Embeddings/EmbeddingsPyTorch.ipynb) lesson: an `EmbeddingBag` layer that averages the vectors of all the words in the text, followed by one linear layer.

The word-to-index mapping comes from the `KeyedVectors` object, and its `vectors` matrix is exactly the weight matrix of an embedding layer. Words that are not in the vocabulary are skipped.

In [ ]:
def encode(df):
    """Convert each article into a list of word indices from the embedding vocabulary."""
    encoded = []
    for title, descr in zip(df.title, df.description):
        ids = [wv.key_to_index[w] for w in tokenize(title + ' ' + descr) if w in wv.key_to_index]
        encoded.append(ids if ids else [0])   # make sure that no text is empty
    return encoded

X_train, y_train = encode(train_df), (train_df.label - 1).values
X_test, y_test = encode(test_df), (test_df.label - 1).values

def make_batches(X, y, batch_size, shuffle):
    order = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
    for i in range(0, len(X), batch_size):
        batch = order[i:i + batch_size]
        texts = [X[j] for j in batch]
        offsets = torch.tensor([0] + [len(t) for t in texts[:-1]]).cumsum(0)
        flat = torch.tensor([i for t in texts for i in t])
        yield flat.to(device), offsets.to(device), torch.tensor(y[batch]).to(device)

class EmbedClassifier(nn.Module):
    def __init__(self, embedding, num_classes=4):
        super().__init__()
        self.embedding = embedding
        self.fc = nn.Linear(embedding.embedding_dim, num_classes)

    def forward(self, x, offsets):
        return self.fc(self.embedding(x, offsets))

We will compare three ways of creating the embedding layer, all with exactly the same vocabulary:

* **random** - the usual random initialization, the embeddings are trained together with the classifier;
* **frozen** - initialized from our skip-gram vectors, and not changed during training;
* **fine-tuned** - initialized from our skip-gram vectors, and then trained further together with the classifier.

Pre-trained embeddings help most when we do not have much labeled data, so we train on small random subsets of the training set (500 and 2000 articles) and evaluate on the whole test set. Notice that the embeddings themselves were trained without using any labels.

In [ ]:
def train_and_evaluate(mode, n_train, epochs=20, lr=5e-3):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if mode == 'random':
        embedding = nn.EmbeddingBag(len(wv), wv.vector_size, mode='mean')
    else:
        embedding = nn.EmbeddingBag.from_pretrained(torch.FloatTensor(wv.vectors),
                                                    freeze=(mode == 'frozen'), mode='mean')
    model = EmbedClassifier(embedding).to(device)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    subset = np.random.RandomState(0).permutation(len(X_train))[:n_train]
    X_sub, y_sub = [X_train[i] for i in subset], y_train[subset]

    for epoch in range(epochs):
        model.train()
        for x, offsets, y in make_batches(X_sub, y_sub, 64, shuffle=True):
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(x, offsets), y)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, offsets, y in make_batches(X_test, y_test, 512, shuffle=False):
            correct += (model(x, offsets).argmax(dim=1) == y).sum().item()
    return correct / len(X_test)

results = {}
for n_train in [500, 2000]:
    results[f'{n_train} training texts'] = {mode: train_and_evaluate(mode, n_train)
                                            for mode in ['random', 'frozen', 'fine-tuned']}
pd.DataFrame(results).T.round(3)   # test accuracy

With only a few labeled examples, the network starting from our embeddings does much better than the network that has to learn the embeddings from scratch. As we get more labeled data, the gap becomes smaller, because the random embeddings can learn what they need from the labels.

> **Note:** The numbers will be a bit different on every run, but the pattern should be the same.

### What about Keras / TensorFlow?

The idea is the same: create an `Embedding` layer and set its weights to the matrix `wv.vectors`. Here is a sketch (we do not run it in this notebook):

```python
import tensorflow as tf

embedding_layer = tf.keras.layers.Embedding(input_dim=len(wv), output_dim=wv.vector_size,
                                            trainable=False)   # use True to fine-tune
embedding_layer.build((None,))
embedding_layer.set_weights([wv.vectors])
```

## Takeaway

* **Gensim** lets us train Word2Vec (`sg=0` for CBoW, `sg=1` for skip-gram) and FastText embeddings in just a few lines of code.
* The most important hyper-parameters are `vector_size`, `window`, `min_count` and `negative`. Embeddings are learned from raw text, so there is no need for labels.
* FastText can build vectors even for words that were not in the training data.
* Embeddings can be saved, loaded and used to initialize an embedding layer of a neural network. This helps a lot when we have little labeled data.

✅ **Challenge:** Train the embeddings with different values of `vector_size`, `window` or `negative`, and check how the analogies and the classification accuracy change. Another idea: replace the news with a different text, such as a book from [Project Gutenberg](https://www.gutenberg.org/), and see which words are similar to each other in a book.